In [2]:
import requests
from bs4 import BeautifulSoup

LIST_URL = "https://card.kbcard.com/CRD/DVIEW/HCAMCXPRICAC0047" #전가맹점

headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://card.kbcard.com/"
}

session = requests.Session()
res = session.get(LIST_URL, headers=headers, timeout=20)
print(res.status_code)
print(res.url)
print(res.text[:1000])

200
https://card.kbcard.com/CRD/DVIEW/HCAMCXPRICAC0047






<!DOCTYPE html>
<html lang="ko">
<head>
	<meta name="naver-site-verification" content="365f59d7b2b5d9b0b943e14b4d7cf4f7ec8ffdfd"/>
	<meta name="google-site-verification" content="wx5GqAlwMOvToSkcMHNDI1Mou1m01Kl62BwJhUiNP1c">
	<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
	<meta http-equiv="X-UA-Compatible" content="IE=edge">
	
	<link rel="shortcut icon" href="https://img1.kbcard.com/LT/images/common/ico/kb.ico" />
	<link rel="stylesheet" type="text/css" href="/CMN/common/pc/css/header_new.css" />
	<link rel="stylesheet" type="text/css" href="/CMN/pc/css/basic.css" />
	<link rel="stylesheet" type="text/css" href="/CMN/pc/css/card.css" />
	<link rel="stylesheet" type="text/css" href="/CMN/pc/css/banner.css" />

	<script type="text/javascript" charset="utf-8" src="/CMN/common/js/lib/jquery-3.4.1.min.js"></script>
	<script type="text/javascript" charset="utf-8" src="/CMN/common/js/lib/jquery-migrate-1.2.1.

In [3]:
soup = BeautifulSoup(res.text, "html.parser")

for a in soup.select("a.linkDetail")[:10]:
    onclick = a.get("onclick", "")
    print(onclick)

javascript:goDetail('09297','')
javascript:goDetail('09297','')
javascript:goDetail('09922','')
javascript:goDetail('09922','')
javascript:goDetail('09570','')
javascript:goDetail('09570','')
javascript:goDetail('09157','')
javascript:goDetail('09157','')
javascript:goDetail('09250','')
javascript:goDetail('09250','')


# 카드 목록 확인

In [4]:
import re
from bs4 import BeautifulSoup

soup = BeautifulSoup(res.text, "html.parser")

card_items = []

for a in soup.select("a.linkDetail"):
    onclick = a.get("onclick", "")
    match = re.search(r"goDetail\('(\d+)',''\)", onclick)
    
    if match:
        code = match.group(1)
        
        title_tag = a.select_one("h3")
        card_name = title_tag.get_text(strip=True) if title_tag else None
        
        card_items.append({
            "card_name": card_name,
            "code": code
        })

for item in card_items[:20]:
    print(item)

{'card_name': 'WE:SH All+ 카드', 'code': '09297'}
{'card_name': 'WE:SH All+ 카드', 'code': '09297'}
{'card_name': 'ALL 카드', 'code': '09922'}
{'card_name': 'ALL 카드', 'code': '09922'}
{'card_name': 'WE:SH Daily 카드', 'code': '09570'}
{'card_name': 'WE:SH Daily 카드', 'code': '09570'}
{'card_name': '가온올림카드(실속형)', 'code': '09157'}
{'card_name': '가온올림카드(실속형)', 'code': '09157'}
{'card_name': 'The Easy카드', 'code': '09250'}
{'card_name': 'The Easy카드', 'code': '09250'}
{'card_name': '마이핏카드(적립형)', 'code': '09247'}
{'card_name': '마이핏카드(적립형)', 'code': '09247'}
{'card_name': 'KB Pay 챌린지카드', 'code': '09113'}
{'card_name': 'KB Pay 챌린지카드', 'code': '09113'}
{'card_name': 'KB Pay 챌린지+카드', 'code': '09114'}
{'card_name': 'KB Pay 챌린지+카드', 'code': '09114'}
{'card_name': 'toss KB국민카드', 'code': '04350'}
{'card_name': 'toss KB국민카드', 'code': '04350'}


In [5]:
unique_cards = []
seen = set()

for item in card_items:
    key = (item["card_name"], item["code"])
    if key not in seen:
        seen.add(key)
        unique_cards.append(item)

print("중복 제거 후 카드 수:", len(unique_cards))
for item in unique_cards[:20]:
    print(item)

중복 제거 후 카드 수: 9
{'card_name': 'WE:SH All+ 카드', 'code': '09297'}
{'card_name': 'ALL 카드', 'code': '09922'}
{'card_name': 'WE:SH Daily 카드', 'code': '09570'}
{'card_name': '가온올림카드(실속형)', 'code': '09157'}
{'card_name': 'The Easy카드', 'code': '09250'}
{'card_name': '마이핏카드(적립형)', 'code': '09247'}
{'card_name': 'KB Pay 챌린지카드', 'code': '09113'}
{'card_name': 'KB Pay 챌린지+카드', 'code': '09114'}
{'card_name': 'toss KB국민카드', 'code': '04350'}


In [6]:
#테스트

target = None

for item in unique_cards:
    if item["card_name"] == "WE:SH All+ 카드":
        target = item
        break

print(target)

{'card_name': 'WE:SH All+ 카드', 'code': '09297'}


In [7]:
detail_url = f"https://card.kbcard.com/CRD/DVIEW/HCAMCXPRICAC0076?mainCC=a&cooperationcode={target['code']}"
print(detail_url)

detail_res = session.get(detail_url, headers=headers, timeout=20)

print(detail_res.status_code)
print(detail_res.url)

for kw in ["WE:SH All+", "주요혜택", "상세혜택", "연회비", "확인사항"]:
    print(kw, "->", kw in detail_res.text)

https://card.kbcard.com/CRD/DVIEW/HCAMCXPRICAC0076?mainCC=a&cooperationcode=09297
200
https://card.kbcard.com/CRD/DVIEW/HCAMCXPRICAC0076?mainCC=a&cooperationcode=09297
WE:SH All+ -> True
주요혜택 -> True
상세혜택 -> True
연회비 -> True
확인사항 -> True


# 상세혜택 수집하기(1번 카드 테스트)

In [8]:
import pandas as pd

tables = pd.read_html(detail_res.text)

print("표 개수:", len(tables))

for i, df in enumerate(tables[:10]):
    print(f"\n===== 표 {i} =====")
    print(df.head())
    print(df.columns)

표 개수: 5

===== 표 0 =====
                                상품서비스 요약 할인율(혜택)               이용실적   할인한도
0                                 국내 가맹점      1%  전월 이용실적 40만원 이상 시     없음
1                                 해외 가맹점      2%  전월 이용실적 40만원 이상 시     없음
2               쇼핑 멤버십 [네이버플러스, 쿠팡 로켓와우]     50%  전월 이용실적 40만원 이상 시  월 5천원
3  OTT [넷플릭스, 유튜브 프리미엄, 웨이브, 티빙, 디즈니플러스]     10%  전월 이용실적 40만원 이상 시  월 5천원
4       이동통신 요금 [SKT, KT, LG U+, Liiv M]      5%  전월 이용실적 40만원 이상 시  월 5천원
Index(['상품서비스 요약', '할인율(혜택)', '이용실적', '할인한도'], dtype='object')

===== 표 1 =====
       구분                                    할인대상
0  쇼핑 멤버십          네이버플러스 멤버십, 쿠팡 로켓와우 멤버십 자동납부요금
1     OTT  넷플릭스, 유튜브 프리미엄, 웨이브, 티빙, 디즈니플러스 자동납부요금
2    이동통신      SKT, KT, LG U+, Liiv M 이동통신 자동납부요금
Index(['구분', '할인대상'], dtype='object')

===== 표 2 =====
     분기   1분기   2분기    3분기      4분기
0   이용월  1~3월  4~6월   7~9월   10~12월
1  적립시기  4월 말  7월 말  10월 말  익년 1월 말
Index(['분기', '1분기', '2분기', '3분기', '4분기'], dtype='object')

===== 표 3 =====
    KB국

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_19988\697939523.py:3: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(detail_res.text)


In [9]:
card_name = target["card_name"]
print(card_name)

WE:SH All+ 카드


In [10]:
benefit_df = tables[0].copy()

benefit_df = benefit_df.rename(columns={
    "상품서비스 요약": "benefit_item",
    "할인율(혜택)": "benefit_rate",
    "이용실적": "performance_condition",
    "할인한도": "benefit_limit"
})

benefit_df["card_name"] = card_name
benefit_df["cooperationcode"] = target["code"]

print(benefit_df.head())

                            benefit_item benefit_rate performance_condition  \
0                                 국내 가맹점           1%     전월 이용실적 40만원 이상 시   
1                                 해외 가맹점           2%     전월 이용실적 40만원 이상 시   
2               쇼핑 멤버십 [네이버플러스, 쿠팡 로켓와우]          50%     전월 이용실적 40만원 이상 시   
3  OTT [넷플릭스, 유튜브 프리미엄, 웨이브, 티빙, 디즈니플러스]          10%     전월 이용실적 40만원 이상 시   
4       이동통신 요금 [SKT, KT, LG U+, Liiv M]           5%     전월 이용실적 40만원 이상 시   

  benefit_limit      card_name cooperationcode  
0            없음  WE:SH All+ 카드           09297  
1            없음  WE:SH All+ 카드           09297  
2         월 5천원  WE:SH All+ 카드           09297  
3         월 5천원  WE:SH All+ 카드           09297  
4         월 5천원  WE:SH All+ 카드           09297  


In [11]:
annual_fee_df = tables[4].copy()

print(annual_fee_df)
print(annual_fee_df.columns)

                                구분    발급유형 기본연회비  제휴연회비     합계
0  국내전용(Local) / 국내외겸용(Mastercard)      일반   7천원  4만8천원  5만5천원
1  국내전용(Local) / 국내외겸용(Mastercard)  모바일 단독   1천원  4만8천원  4만9천원
Index(['구분', '발급유형', '기본연회비', '제휴연회비', '합계'], dtype='object')


In [12]:
# 1. 혜택 표 정리
benefit_df = tables[0].copy()

benefit_df = benefit_df.rename(columns={
    "상품서비스 요약": "benefit_item",
    "할인율(혜택)": "benefit_rate",
    "이용실적": "performance_condition",
    "할인한도": "benefit_limit"
})

benefit_df["card_name"] = card_name
benefit_df["cooperationcode"] = target["code"]

print("=== 혜택 표 ===")
print(benefit_df.head())


# 2. 연회비 표 정리
annual_fee_df = tables[4].copy()
annual_fee_df["card_name"] = card_name
annual_fee_df["cooperationcode"] = target["code"]

print("\n=== 연회비 표 ===")
print(annual_fee_df.head())


# 3. 저장
benefit_df.to_csv("kbcard_benefit_test.csv", index=False, encoding="utf-8-sig")
annual_fee_df.to_csv("kbcard_annual_fee_test.csv", index=False, encoding="utf-8-sig")

print("\n저장 완료")

=== 혜택 표 ===
                            benefit_item benefit_rate performance_condition  \
0                                 국내 가맹점           1%     전월 이용실적 40만원 이상 시   
1                                 해외 가맹점           2%     전월 이용실적 40만원 이상 시   
2               쇼핑 멤버십 [네이버플러스, 쿠팡 로켓와우]          50%     전월 이용실적 40만원 이상 시   
3  OTT [넷플릭스, 유튜브 프리미엄, 웨이브, 티빙, 디즈니플러스]          10%     전월 이용실적 40만원 이상 시   
4       이동통신 요금 [SKT, KT, LG U+, Liiv M]           5%     전월 이용실적 40만원 이상 시   

  benefit_limit      card_name cooperationcode  
0            없음  WE:SH All+ 카드           09297  
1            없음  WE:SH All+ 카드           09297  
2         월 5천원  WE:SH All+ 카드           09297  
3         월 5천원  WE:SH All+ 카드           09297  
4         월 5천원  WE:SH All+ 카드           09297  

=== 연회비 표 ===
                                구분    발급유형 기본연회비  제휴연회비     합계      card_name  \
0  국내전용(Local) / 국내외겸용(Mastercard)      일반   7천원  4만8천원  5만5천원  WE:SH All+ 카드   
1  국내전용(Local) / 국내외겸용(Mastercard)  모바일 단

# 카드별 혜택 수집하기

In [13]:
import re
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://card.kbcard.com/"
}

session = requests.Session()

# 카드 목록 페이지 요청
LIST_URL = "https://card.kbcard.com/CRD/DVIEW/HCAMCXPRICAC0047"
res = session.get(LIST_URL, headers=headers, timeout=20)
res.raise_for_status()

soup = BeautifulSoup(res.text, "html.parser")

# 카드명 + 코드 추출
card_items = []
for a in soup.select("a.linkDetail"):
    onclick = a.get("onclick", "")
    match = re.search(r"goDetail\('(\d+)',''\)", onclick)

    if match:
        code = match.group(1)
        title_tag = a.select_one("h3")
        card_name = title_tag.get_text(strip=True) if title_tag else None

        card_items.append({
            "card_name": card_name,
            "code": code
        })

# 중복 제거
unique_cards = []
seen = set()

for item in card_items:
    key = (item["card_name"], item["code"])
    if key not in seen:
        seen.add(key)
        unique_cards.append(item)

print("수집 대상 카드 수:", len(unique_cards))
for item in unique_cards[:10]:
    print(item)

수집 대상 카드 수: 9
{'card_name': 'WE:SH All+ 카드', 'code': '09297'}
{'card_name': 'ALL 카드', 'code': '09922'}
{'card_name': 'WE:SH Daily 카드', 'code': '09570'}
{'card_name': '가온올림카드(실속형)', 'code': '09157'}
{'card_name': 'The Easy카드', 'code': '09250'}
{'card_name': '마이핏카드(적립형)', 'code': '09247'}
{'card_name': 'KB Pay 챌린지카드', 'code': '09113'}
{'card_name': 'KB Pay 챌린지+카드', 'code': '09114'}
{'card_name': 'toss KB국민카드', 'code': '04350'}


In [15]:
def parse_card_detail(card_name, code, session, headers):
    detail_url = f"https://card.kbcard.com/CRD/DVIEW/HCAMCXPRICAC0076?mainCC=a&cooperationcode={code}"

    r = session.get(detail_url, headers=headers, timeout=20)
    r.raise_for_status()

    tables = pd.read_html(r.text)

    benefit_df = None
    annual_fee_df = None

    # 혜택 표 찾기
    for t in tables:
        cols = list(t.columns)
        if all(col in cols for col in ["상품서비스 요약", "할인율(혜택)", "이용실적", "할인한도"]):
            benefit_df = t.copy().rename(columns={
                "상품서비스 요약": "benefit_item",
                "할인율(혜택)": "benefit_rate",
                "이용실적": "performance_condition",
                "할인한도": "benefit_limit"
            })
            benefit_df["card_name"] = card_name
            benefit_df["cooperationcode"] = code
            benefit_df = benefit_df[
                ["card_name", "cooperationcode", "benefit_item", "benefit_rate", "performance_condition", "benefit_limit"]
            ]
            break

    # 연회비 표 찾기
    for t in tables:
        cols = list(t.columns)
        if all(col in cols for col in ["구분", "발급유형", "기본연회비", "제휴연회비", "합계"]):
            annual_fee_df = t.copy().rename(columns={
                "구분": "card_scope",
                "발급유형": "issue_type",
                "기본연회비": "base_fee",
                "제휴연회비": "partner_fee",
                "합계": "total_fee"
            })
            annual_fee_df["card_name"] = card_name
            annual_fee_df["cooperationcode"] = code
            annual_fee_df = annual_fee_df[
                ["card_name", "cooperationcode", "card_scope", "issue_type", "base_fee", "partner_fee", "total_fee"]
            ]
            break

    return benefit_df, annual_fee_df, detail_url

In [16]:
print(parse_card_detail)

<function parse_card_detail at 0x0000020C047E5EA0>


In [17]:
all_benefits = []
all_fees = []
errors = []

for idx, item in enumerate(unique_cards, start=1):
    card_name = item["card_name"]
    code = item["code"]

    try:
        benefit_df, annual_fee_df, detail_url = parse_card_detail(card_name, code, session, headers)

        print(f"[{idx}/{len(unique_cards)}] {card_name} ({code})")

        if benefit_df is not None:
            all_benefits.append(benefit_df)

        if annual_fee_df is not None:
            all_fees.append(annual_fee_df)

        time.sleep(0.5)

    except Exception as e:
        print(f"에러 발생: {card_name} ({code}) -> {e}")
        errors.append({
            "card_name": card_name,
            "code": code,
            "error": str(e)
        })

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_19988\1309222926.py:7: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(r.text)


[1/9] WE:SH All+ 카드 (09297)


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_19988\1309222926.py:7: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(r.text)


[2/9] ALL 카드 (09922)


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_19988\1309222926.py:7: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(r.text)


...

[7/9] KB Pay 챌린지카드 (09113)


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_19988\1309222926.py:7: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(r.text)


[8/9] KB Pay 챌린지+카드 (09114)


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_19988\1309222926.py:7: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(r.text)


[9/9] toss KB국민카드 (04350)


In [40]:
if all_benefits:
    benefits_final = pd.concat(all_benefits, ignore_index=True)
    print("혜택 최종 shape:", benefits_final.shape)
    print("혜택 카드 종류 수:", benefits_final["card_name"].nunique())
    print(benefits_final["card_name"].unique())

    benefits_final.to_csv("kbcard_benefits.csv", index=False, encoding="utf-8-sig")
    print("혜택 테이블 저장 완료")

if all_fees:
    fees_final = pd.concat(all_fees, ignore_index=True)
    print("연회비 최종 shape:", fees_final.shape)
    print("연회비 카드 종류 수:", fees_final["card_name"].nunique())
    print(fees_final["card_name"].unique())

    fees_final.to_csv("kbcard_annual_fees.csv", index=False, encoding="utf-8-sig")
    print("연회비 테이블 저장 완료")

혜택 최종 shape: (7, 6)
혜택 카드 종류 수: 1
['WE:SH All+ 카드']
혜택 테이블 저장 완료
연회비 최종 shape: (6, 7)
연회비 카드 종류 수: 3
['WE:SH All+ 카드' 'ALL 카드' 'WE:SH Daily 카드']
연회비 테이블 저장 완료


In [41]:
df.shape

(7, 6)

In [37]:
df = pd.read_csv("./kbcard_benefits.csv")
df

,card_name,cooperationcode,benefit_item,benefit_rate,performance_condition,benefit_limit
0,WE:SH All+ 카드,9297,국내 가맹점,1%,전월 이용실적 40만원 이상 시,없음
1,WE:SH All+ 카드,9297,해외 가맹점,2%,전월 이용실적 40만원 이상 시,없음
2,WE:SH All+ 카드,9297,"쇼핑 멤버십 [네이버플러스, 쿠팡 로켓와우]",50%,전월 이용실적 40만원 이상 시,월 5천원
3,WE:SH All+ 카드,9297,"OTT [넷플릭스, 유튜브 프리미엄, 웨이브, 티빙, 디즈니플러스]",10%,전월 이용실적 40만원 이상 시,월 5천원
4,WE:SH All+ 카드,9297,"이동통신 요금 [SKT, KT, LG U+, Liiv M]",5%,전월 이용실적 40만원 이상 시,월 5천원
5,WE:SH All+ 카드,9297,분기 보너스 적립,1만점,분기 이용실적 400만원 이상 시,분기 1회
6,WE:SH All+ 카드,9297,"Mastercard 티타늄 등급 서비스 [호텔/공항 발레파킹, 공항라운지 무료이용 등]",-,[기본서비스] 없음 [선택서비스] 전월 이용실적 40만원 이상 시,서비스별 상이


In [39]:
len(all_benefits), len(all_fees), len(errors)

(1, 3, 0)

In [43]:
from bs4 import BeautifulSoup

detail_soup = BeautifulSoup(detail_res.text, "html.parser")

service_div = detail_soup.select_one("#tabConD10")
print(service_div is not None)

False


In [46]:
"tabConD" in detail_res.text

False

In [47]:
errors

[]

# PDF파일로 수집하기

In [49]:
".pdf" in detail_res.text

True

In [18]:
import re

pdf_links = re.findall(r'https://[^\s"\']+\.pdf', detail_res.text)
pdf_links

['https://img2.kbcard.com/obj/card/download/A0027_stpul_20250328.pdf',
 'https://img2.kbcard.com/obj/card/download/A0028_stpul_20251128.pdf',
 'https://img2.kbcard.com/obj/card/download/09297__prdctOpmn_20260327.pdf',
 'https://img2.kbcard.com/obj/contents/download/Terms_sheet_persnal_MERGE_09297.pdf']

In [19]:
import re

def extract_product_pdf_link(html):
    pdf_links = re.findall(r'https://[^\s"\']+\.pdf', html)
    
    for link in pdf_links:
        if "prdctOpmn" in link:
            return link
    
    return None

pdf_link = extract_product_pdf_link(detail_res.text)
print(pdf_link)

https://img2.kbcard.com/obj/card/download/09297__prdctOpmn_20260327.pdf


In [21]:
import requests
import re
import pandas as pd

def extract_product_pdf_link(html):
    pdf_links = re.findall(r'https://[^\s"\']+\.pdf', html)
    for link in pdf_links:
        if "prdctOpmn" in link:
            return link
    return None


def collect_card_pdfs(card_df):
    results = []

    session = requests.Session()

    for i, row in card_df.iterrows():
        code = row["cooperationcode"]
        name = row["card_name"]

        detail_url = f"https://card.kbcard.com/CRD/DVIEW/HCAMCXPRICAC0076?mainCC=a&cooperationcode={code}"

        try:
            res = session.get(detail_url, timeout=10)
            res.raise_for_status()

            pdf_link = extract_product_pdf_link(res.text)

            results.append({
                "card_name": name,
                "cooperationcode": code,
                "pdf_link": pdf_link
            })

            print(f"{name} 완료")

        except Exception as e:
            print(f" {name} 실패:", e)

    return pd.DataFrame(results)

In [22]:
card_df = pd.DataFrame(unique_cards)
card_df

,card_name,code
0,WE:SH All+ 카드,09297
1,ALL 카드,09922
2,WE:SH Daily 카드,09570
3,가온올림카드(실속형),09157
4,The Easy카드,09250
5,마이핏카드(적립형),09247
6,KB Pay 챌린지카드,09113
7,KB Pay 챌린지+카드,09114
8,toss KB국민카드,04350


In [23]:
card_df = pd.DataFrame(unique_cards).rename(columns={"code": "cooperationcode"})
card_df

,card_name,cooperationcode
0,WE:SH All+ 카드,09297
1,ALL 카드,09922
2,WE:SH Daily 카드,09570
3,가온올림카드(실속형),09157
4,The Easy카드,09250
5,마이핏카드(적립형),09247
6,KB Pay 챌린지카드,09113
7,KB Pay 챌린지+카드,09114
8,toss KB국민카드,04350


In [24]:
pdf_df = collect_card_pdfs(card_df)
pdf_df

WE:SH All+ 카드 완료
ALL 카드 완료
WE:SH Daily 카드 완료
가온올림카드(실속형) 완료
The Easy카드 완료
마이핏카드(적립형) 완료
KB Pay 챌린지카드 완료
KB Pay 챌린지+카드 완료
toss KB국민카드 완료


,card_name,cooperationcode,pdf_link
0,WE:SH All+ 카드,09297,https://img2.kbcard.com/obj/card/download/0929...
1,ALL 카드,09922,https://img2.kbcard.com/obj/card/download/_099...
2,WE:SH Daily 카드,09570,https://img2.kbcard.com/obj/card/download/0957...
3,가온올림카드(실속형),09157,https://img2.kbcard.com/obj/card/download/0915...
4,The Easy카드,09250,https://img2.kbcard.com/obj/card/download/0925...
5,마이핏카드(적립형),09247,https://img2.kbcard.com/obj/card/download/0924...
6,KB Pay 챌린지카드,09113,https://img2.kbcard.com/obj/card/download/0911...
7,KB Pay 챌린지+카드,09114,https://img2.kbcard.com/obj/card/download/0911...
8,toss KB국민카드,04350,https://img2.kbcard.com/obj/card/download/0435...


## 카드 2~3개 테스트

In [61]:
# !pip install pdfplumber

In [65]:
import re

def clean_filename(name):
    return re.sub(r'[\\/:*?"<>|]', '_', name)

In [66]:
import os
import requests

os.makedirs("kb_pdfs", exist_ok=True)

for _, row in pdf_df.iterrows():
    url = row["pdf_link"]
    card_name = row["card_name"]
    code = row["cooperationcode"]

    safe_name = clean_filename(card_name)

    file_path = f"kb_pdfs/{code}_{safe_name}.pdf"

    r = requests.get(url)
    with open(file_path, "wb") as f:
        f.write(r.content)

    print("저장 완료:", file_path)

저장 완료: kb_pdfs/09297_WE_SH All+ 카드.pdf
저장 완료: kb_pdfs/09922_ALL 카드.pdf
저장 완료: kb_pdfs/09570_WE_SH Daily 카드.pdf
저장 완료: kb_pdfs/09157_가온올림카드(실속형).pdf
저장 완료: kb_pdfs/09250_The Easy카드.pdf
저장 완료: kb_pdfs/09247_마이핏카드(적립형).pdf
저장 완료: kb_pdfs/09113_KB Pay 챌린지카드.pdf
저장 완료: kb_pdfs/09114_KB Pay 챌린지+카드.pdf
저장 완료: kb_pdfs/04350_toss KB국민카드.pdf


In [68]:
def extract_pdf_text(pdf_path):
    texts = []
    
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                texts.append(text)
    
    return "\n".join(texts)

In [69]:
def clean_text(text):
    if not text:
        return ""
    
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()

In [70]:
def extract_core_fields(text, fallback_card_name=None):
    result = {
        "card_name": fallback_card_name,
        "annual_fee_summary": None,
        "previous_month_spend_summary": None,
        "monthly_limit_summary": None,
        "benefits_summary": None
    }
    
    # 연회비 관련 문장
    annual_fee_matches = re.findall(r".{0,20}(연회비).{0,60}", text)
    if annual_fee_matches:
        lines = [line.strip() for line in text.split("\n") if "연회비" in line]
        result["annual_fee_summary"] = " | ".join(lines[:5]) if lines else None
    
    # 전월실적 관련 문장
    spend_lines = [line.strip() for line in text.split("\n") if ("전월" in line and "실적" in line) or "이용실적" in line]
    if spend_lines:
        result["previous_month_spend_summary"] = " | ".join(spend_lines[:5])
    
    # 월 한도 관련 문장
    limit_lines = [
        line.strip() for line in text.split("\n")
        if ("월" in line and ("한도" in line or "적립" in line or "할인" in line))
    ]
    if limit_lines:
        result["monthly_limit_summary"] = " | ".join(limit_lines[:5])
    
    # 혜택 관련 문장
    benefit_lines = [
        line.strip() for line in text.split("\n")
        if any(keyword in line for keyword in ["할인", "적립", "서비스", "혜택"])
    ]
    if benefit_lines:
        result["benefits_summary"] = " | ".join(benefit_lines[:10])
    
    return result

In [71]:
def extract_tables_from_pdf(pdf_path):
    all_tables = []
    
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            tables = page.extract_tables()
            if not tables:
                continue
            
            for table_idx, table in enumerate(tables, start=1):
                if not table or len(table) < 2:
                    continue
                
                try:
                    df = pd.DataFrame(table[1:], columns=table[0])
                    df["page_num"] = page_num
                    df["table_idx"] = table_idx
                    all_tables.append(df)
                except Exception:
                    continue
    
    return all_tables

In [72]:
def classify_table(df):
    col_text = " ".join([str(c) for c in df.columns])
    body_text = " ".join(df.astype(str).fillna("").head(10).values.flatten())
    full_text = col_text + " " + body_text
    
    if any(k in full_text for k in ["연회비", "기본연회비", "제휴연회비", "합계"]):
        return "annual_fee"
    
    if any(k in full_text for k in ["할인", "적립", "이용실적", "한도", "서비스", "혜택"]):
        return "benefit"
    
    return "other"

In [73]:
def parse_card_pdf(pdf_path, card_name=None, cooperationcode=None, pdf_link=None):
    raw_text = extract_pdf_text(pdf_path)
    text = clean_text(raw_text)
    
    summary = extract_core_fields(text, fallback_card_name=card_name)
    summary["cooperationcode"] = cooperationcode
    summary["pdf_link"] = pdf_link
    summary["pdf_path"] = pdf_path
    
    summary_df = pd.DataFrame([summary])
    
    table_dfs = extract_tables_from_pdf(pdf_path)
    
    benefit_tables = []
    fee_tables = []
    
    for df in table_dfs:
        table_type = classify_table(df)
        
        df = df.copy()
        df["card_name"] = card_name
        df["cooperationcode"] = cooperationcode
        df["pdf_path"] = pdf_path
        
        if table_type == "benefit":
            benefit_tables.append(df)
        elif table_type == "annual_fee":
            fee_tables.append(df)
    
    return summary_df, benefit_tables, fee_tables, text

In [75]:
#카드 1개 테스트
sample_row = pdf_df.iloc[0]

safe_name = re.sub(r'[\\/:*?"<>|]', '_', sample_row['card_name'])
sample_pdf_path = f"kb_pdfs/{sample_row['cooperationcode']}_{safe_name}.pdf"

summary_df, benefit_tables, fee_tables, full_text = parse_card_pdf(
    pdf_path=sample_pdf_path,
    card_name=sample_row["card_name"],
    cooperationcode=sample_row["cooperationcode"],
    pdf_link=sample_row["pdf_link"]
)

print(summary_df.T)
print("혜택표 개수:", len(benefit_tables))
print("연회비표 개수:", len(fee_tables))
print(full_text[:2000])

                                                                              0
card_name                                                         WE:SH All+ 카드
annual_fee_summary            강/연금/고용/산재), 각종수수료및이자, 연체료, 연회비, 신차 | 각종수수료및이자...
previous_month_spend_summary  ※ 할인한도없음(전월이용실적40만원이상시제공) ∙ 분기실적400만원이상이용시1만점포...
monthly_limit_summary         ※ 할인한도없음(전월이용실적40만원이상시제공) ∙ 분기실적400만원이상이용시1만점포...
benefits_summary              언제 어디서나 쉬운 할인 | 더 많은 혜택을 더한 ‘모두’의 카드 | 할인 서비스 ...
cooperationcode                                                           09297
pdf_link                      https://img2.kbcard.com/obj/card/download/0929...
pdf_path                                        kb_pdfs/09297_WE_SH All+ 카드.pdf
혜택표 개수: 0
연회비표 개수: 0
(상품설명서) A-20250331-9641–00722-00
언제 어디서나 쉬운 할인
더 많은 혜택을 더한 ‘모두’의 카드
KB WE:SH All+ 카드
위시 올 플러스 카드
할인 서비스 특별 서비스
국내 가맹점 1% 할인 분기 보너스 적립 (연 최대 4만점)
※ 할인한도없음(전월이용실적40만원이상시제공) ∙ 분기실적400만원이상이용시1만점포인트리 적립
※ 최초발급받은KB WE:SH All+카드사용등록일(KB Pay 등간편결 (분기별1만점)
제등록포함)로부터다음

In [76]:
# 카드 전체 수집
all_summary = []
all_benefit_tables = []
all_fee_tables = []
parse_errors = []

for _, row in pdf_df.iterrows():
    try:
        safe_name = re.sub(r'[\\/:*?"<>|]', '_', row["card_name"])
        pdf_path = f"kb_pdfs/{row['cooperationcode']}_{safe_name}.pdf"
        
        summary_df, benefit_tables, fee_tables, full_text = parse_card_pdf(
            pdf_path=pdf_path,
            card_name=row["card_name"],
            cooperationcode=row["cooperationcode"],
            pdf_link=row["pdf_link"]
        )
        
        all_summary.append(summary_df)
        all_benefit_tables.extend(benefit_tables)
        all_fee_tables.extend(fee_tables)
        
        print(f"완료: {row['card_name']}")
    
    except Exception as e:
        print(f"실패: {row['card_name']} -> {e}")
        parse_errors.append({
            "card_name": row["card_name"],
            "cooperationcode": row["cooperationcode"],
            "error": str(e)
        })

완료: WE:SH All+ 카드
완료: ALL 카드
완료: WE:SH Daily 카드
완료: 가온올림카드(실속형)
완료: The Easy카드
완료: 마이핏카드(적립형)
완료: KB Pay 챌린지카드
완료: KB Pay 챌린지+카드
완료: toss KB국민카드


In [77]:
if all_summary:
    summary_final = pd.concat(all_summary, ignore_index=True)
    summary_final.to_csv("kbcard_pdf_summary.csv", index=False, encoding="utf-8-sig")
    print("요약 CSV 저장 완료:", summary_final.shape)

if all_benefit_tables:
    benefit_final = pd.concat(all_benefit_tables, ignore_index=True)
    benefit_final.to_csv("kbcard_pdf_benefit_tables.csv", index=False, encoding="utf-8-sig")
    print("혜택표 CSV 저장 완료:", benefit_final.shape)

if all_fee_tables:
    fee_final = pd.concat(all_fee_tables, ignore_index=True)
    fee_final.to_csv("kbcard_pdf_fee_tables.csv", index=False, encoding="utf-8-sig")
    print("연회비표 CSV 저장 완료:", fee_final.shape)

if parse_errors:
    error_df = pd.DataFrame(parse_errors)
    error_df.to_csv("kbcard_pdf_parse_errors.csv", index=False, encoding="utf-8-sig")
    print("에러 CSV 저장 완료:", error_df.shape)

요약 CSV 저장 완료: (9, 8)
혜택표 CSV 저장 완료: (10, 16)


In [78]:
read = pd.read_csv("./kbcard_pdf_summary.csv")
read

,card_name,annual_fee_summary,previous_month_spend_summary,monthly_limit_summary,benefits_summary,cooperationcode,pdf_link,pdf_path
0,WE:SH All+ 카드,"강/연금/고용/산재), 각종수수료및이자, 연체료, 연회비, 신차 | 각종수수료및이자...",※ 할인한도없음(전월이용실적40만원이상시제공) ∙ 분기실적400만원이상이용시1만점포...,※ 할인한도없음(전월이용실적40만원이상시제공) ∙ 분기실적400만원이상이용시1만점포...,언제 어디서나 쉬운 할인 | 더 많은 혜택을 더한 ‘모두’의 카드 | 할인 서비스 ...,9297,https://img2.kbcard.com/obj/card/download/0929...,kb_pdfs/09297_WE_SH All+ 카드.pdf
1,ALL 카드,연회비,할인 서비스 제외 대상 전월 이용실적 제외 대상 | 전월 이용실적 기준,할인 서비스 제외 대상 전월 이용실적 제외 대상,할인 서비스 | 구분 할인대상 | 할인 서비스 제외 대상 전월 이용실적 제외 대상 ...,9922,https://img2.kbcard.com/obj/card/download/_099...,kb_pdfs/09922_ALL 카드.pdf
2,WE:SH Daily 카드,NaN,NaN,NaN,데일리스탬프모으면늘어나는혜택,9570,https://img2.kbcard.com/obj/card/download/0957...,kb_pdfs/09570_WE_SH Daily 카드.pdf
3,가온올림카드(실속형),"수수료및이자, 연체료, 연회비, 상품권및선불카드 구입 | 각종수수료및이자, 연체료,...","가온올림카드(실속형) 기본/추가적립서비스, 해외이용할인캐시백은전월실적조건없이제공됩니...","가온올림카드(실속형) 기본/추가적립서비스, 해외이용할인캐시백은전월실적조건없이제공됩니...","가온올림카드(실속형) 기본/추가적립서비스, 해외이용할인캐시백은전월실적조건없이제공됩니...",9157,https://img2.kbcard.com/obj/card/download/0915...,kb_pdfs/09157_가온올림카드(실속형).pdf
4,The Easy카드,"이자, 연체료, 연회비, 상품권 및 선불카드(선불전자지급수단포함) 구입∙충전금액 |...",∙전월 실적조건 및 월 적립한도 없음 ∙전월 실적조건 및 월 할인한도 없음 | 구분...,∙적립 시점 기준 60개월이 경과한 포인트리는 적립순서에 따라 월 | ∙전월 실적조...,"적립과 할인을 | ∙적립, 할인형 선택은 KB국민카드 모바일앱, 홈페이지 | 가능하...",9250,https://img2.kbcard.com/obj/card/download/0925...,kb_pdfs/09250_The Easy카드.pdf
5,마이핏카드(적립형),연회비 | 3만원(기본연회비1천원+ 제휴연회비2만9천원) | • 초회/차기년도연회비...,"마이핏카드(적립형)서비스는전월이용실적50만원이상시제공되며, 전월이용실적에따라월간적립...","마이핏카드(적립형)서비스는전월이용실적50만원이상시제공되며, 전월이용실적에따라월간적립...",마이핏카드(적립형) | ‘KB국민마이핏카드(적립형)’는실물없는모바일단독카드로KBPa...,9247,https://img2.kbcard.com/obj/card/download/0924...,kb_pdfs/09247_마이핏카드(적립형).pdf
6,KB Pay 챌린지카드,"보험료(건강/연금/고용/산재), 각종수수료및이자, 연체료, 연회비, 상품권및선불카드...",NaN,-월간적립한도5만점 -이용건당1만원이상이용에한함 | ◦월간적립한도적용기간 -포인트리...,포인트적립되는재미가쌓인다! | 스탬프서비스 출석체크서비스 | · KB Pay 3회이...,9113,https://img2.kbcard.com/obj/card/download/0911...,kb_pdfs/09113_KB Pay 챌린지카드.pdf
7,KB Pay 챌린지+카드,"등록금, 4대사회보험료(건강/연금/고용/산재), 각종수수료및이자, 연체료, 연회비,...",NaN,-월간적립한도5만점 -이용건당1만원이상이용에한함 | ◦월간적립한도적용기간 -포인트리...,포인트적립되는재미가쌓인다! | √ 스탬프서비스 √ 출석체크서비스 | · KB Pay...,9114,https://img2.kbcard.com/obj/card/download/0911...,kb_pdfs/09114_KB Pay 챌린지+카드.pdf
8,toss KB국민카드,"연체료, 연회비, 상품권및선불가드(선불전자지급수단포함) 구입·충전금액, 4대사회보험...",※ 토스포인트기본적립서비스는전월실적조건및적립한도가없습니다. | 전월이용실적조건 적립...,※ 토스포인트기본적립서비스는전월실적조건및적립한도가없습니다. | 전월이용실적조건 적립...,심플하고 쉽게 토스포인트 적립하는 방법! | 토스포인트 기본 적립+ 추가 적립까지!...,4350,https://img2.kbcard.com/obj/card/download/0435...,kb_pdfs/04350_toss KB국민카드.pdf


In [79]:
benefit = pd.read_csv("./kbcard_pdf_benefit_tables.csv")
benefit

,구분,적립률,전월 이용 실적,월 적립한도,page_num,table_idx,card_name,cooperationcode,pdf_path,할인율,월 할인한도,전월이용실적,월적립한도,제공조건,전월이용실적조건,확인사항
0,추가적립,3%,50만원 이상\n100만원 미만,1만점,2,1,The Easy카드,9250,kb_pdfs/09250_The Easy카드.pdf,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,100만원 이상,2만점,2,1,The Easy카드,9250,kb_pdfs/09250_The Easy카드.pdf,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,추가할인,NaN,50만원 이상\n100만원 미만,NaN,2,2,The Easy카드,9250,kb_pdfs/09250_The Easy카드.pdf,5%,1만원,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,100만원 이상,NaN,2,2,The Easy카드,9250,kb_pdfs/09250_The Easy카드.pdf,NaN,2만원,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,1,1,마이핏카드(적립형),9247,kb_pdfs/09247_마이핏카드(적립형).pdf,NaN,NaN,50만원이상,3만점,- 건당2만원이상이용시\n- 일적립한도1만점,NaN,NaN
5,NaN,NaN,NaN,NaN,1,2,마이핏카드(적립형),9247,kb_pdfs/09247_마이핏카드(적립형).pdf,NaN,NaN,50만원이상100만원미만,3천점,- 건당1만원이상이용시,NaN,NaN
6,NaN,NaN,NaN,NaN,1,2,마이핏카드(적립형),9247,kb_pdfs/09247_마이핏카드(적립형).pdf,NaN,NaN,100만원이상,5천점,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,1,3,마이핏카드(적립형),9247,kb_pdfs/09247_마이핏카드(적립형).pdf,NaN,NaN,50만원이상100만원미만,3천점,- 건당1만원이상이용시,NaN,NaN
8,NaN,NaN,NaN,NaN,1,3,마이핏카드(적립형),9247,kb_pdfs/09247_마이핏카드(적립형).pdf,NaN,NaN,100만원이상,5천점,NaN,NaN,NaN
9,NaN,4.0%,NaN,NaN,1,1,toss KB국민카드,4350,kb_pdfs/04350_toss KB국민카드.pdf,NaN,NaN,NaN,5천점,NaN,NaN,NaN


## 2차 시도

# 2차 시도

In [25]:
import requests
import pdfplumber
import pandas as pd
import time
import re
from io import BytesIO

# ==============================
# 1. PDF 텍스트 추출
# ==============================
def extract_pdf_text(pdf_url, max_pages=5):
    try:
        res = requests.get(pdf_url, timeout=20)
        res.raise_for_status()

        texts = []
        with pdfplumber.open(BytesIO(res.content)) as pdf:
            pages = pdf.pages[:max_pages]

            for page in pages:
                text = page.extract_text()
                if text:
                    texts.append(text)

        return "\n".join(texts)

    except Exception as e:
        print(f"❌ PDF 읽기 실패: {pdf_url} / {e}")
        return ""


# ==============================
# 2. 연회비 추출
# ==============================
def extract_fee(text):
    pattern = r"\d{1,3}(,\d{3})*원"
    matches = re.findall(pattern, text)

    if matches:
        return list(set(matches))
    return []


# ==============================
# 3. 전월실적 추출
# ==============================
def extract_spend(text):
    lines = text.split("\n")
    result = []

    for line in lines:
        if "전월" in line and "실적" in line:
            result.append(line.strip())

    return result[:5]


# ==============================
# 4. 월한도 추출
# ==============================
def extract_limit(text):
    lines = text.split("\n")
    result = []

    for line in lines:
        if "한도" in line:
            result.append(line.strip())

    return result[:5]


# ==============================
# 5. 혜택 추출
# ==============================
def extract_benefits(text):
    lines = text.split("\n")
    result = []

    for line in lines:
        if any(keyword in line for keyword in ["적립", "할인", "%"]):
            result.append(line.strip())

    return result[:10]


# ==============================
# 6. 전체 파이프라인
# ==============================
def process_cards(pdf_df):
    results = []

    for idx, row in pdf_df.iterrows():
        card_name = row["card_name"]
        pdf_url = row["pdf_link"]

        print(f"🔍 처리중: {card_name}")

        text = extract_pdf_text(pdf_url)

        if not text:
            continue

        result = {
            "card_name": card_name,
            "pdf_link": pdf_url,
            "fee": extract_fee(text),
            "spend_condition": extract_spend(text),
            "limit": extract_limit(text),
            "benefits": extract_benefits(text),
        }

        results.append(result)

        time.sleep(0.5)  # 서버 부하 방지

    return pd.DataFrame(results)


# ==============================
# 7. 실행
# ==============================
# 예시: pdf_df 구조
# pdf_df = pd.DataFrame({
#     "card_name": ["토스 KB국민카드"],
#     "pdf_link": ["https://img2.kbcard.com/...pdf"]
# })

result_df = process_cards(pdf_df)

# 저장
result_df.to_csv("kbcard_parsed.csv", index=False, encoding="utf-8-sig")

result_df.head()

🔍 처리중: WE:SH All+ 카드
🔍 처리중: ALL 카드
🔍 처리중: WE:SH Daily 카드
🔍 처리중: 가온올림카드(실속형)
🔍 처리중: The Easy카드
🔍 처리중: 마이핏카드(적립형)
🔍 처리중: KB Pay 챌린지카드
🔍 처리중: KB Pay 챌린지+카드
🔍 처리중: toss KB국민카드


,card_name,pdf_link,fee,spend_condition,limit,benefits
0,WE:SH All+ 카드,https://img2.kbcard.com/obj/card/download/0929...,"[,500, ,000]",[※ 할인한도없음(전월이용실적40만원이상시제공) ∙ 분기실적400만원이상이용시1만점...,[※ 할인한도없음(전월이용실적40만원이상시제공) ∙ 분기실적400만원이상이용시1만점...,"[언제 어디서나 쉬운 할인, 할인 서비스 특별 서비스, 국내 가맹점 1% 할인 분기..."
1,ALL 카드,https://img2.kbcard.com/obj/card/download/_099...,[],"[할인 서비스 제외 대상 전월 이용실적 제외 대상, 전월 이용실적 기준]",[],"[할인 서비스, 구분 할인대상, 할인 서비스 제외 대상 전월 이용실적 제외 대상]"
2,WE:SH Daily 카드,https://img2.kbcard.com/obj/card/download/0957...,[],[],[],[]
3,가온올림카드(실속형),https://img2.kbcard.com/obj/card/download/0915...,"[,500]","[가온올림카드(실속형) 기본/추가적립서비스, 해외이용할인캐시백은전월실적조건없이제공됩...",[이용한도등에영향을미칠수있습니다.],"[가온올림카드(실속형) 기본/추가적립서비스, 해외이용할인캐시백은전월실적조건없이제공됩..."
4,The Easy카드,https://img2.kbcard.com/obj/card/download/0925...,[],"[∙전월 실적조건 및 월 적립한도 없음 ∙전월 실적조건 및 월 할인한도 없음, 구분...","[∙전월 실적조건 및 월 적립한도 없음 ∙전월 실적조건 및 월 할인한도 없음, 구분...","[적립과 할인을, ∙적립, 할인형 선택은 KB국민카드 모바일앱, 홈페이지, ∙적립,..."


# HTML + PDF 

In [26]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO

In [43]:
from bs4 import BeautifulSoup, Tag

def extract_detail_section_before_product_desc(html):
    soup = BeautifulSoup(html, "html.parser")
    
    container = soup.select_one("div.contentArea.contentDetail")
    if container is None:
        return None, None
    
    new_soup = BeautifulSoup("<div id='detail_only'></div>", "html.parser")
    target_div = new_soup.select_one("#detail_only")
    
    for child in container.children:
        if isinstance(child, Tag):
            text = child.get_text(" ", strip=True)
            
            # 상품설명서 나오면 중단
            if child.name == "h3" and "상품설명서" in text:
                break
            
            target_div.append(child)
    
    return str(target_div), target_div.get_text("\n", strip=True)

In [44]:
#잘린 구간에서 표 추출
import pandas as pd
from io import StringIO

def extract_tables_from_detail_html(detail_html, card_name, code, detail_url):
    if not detail_html:
        return [], []
    
    try:
        tables = pd.read_html(StringIO(detail_html))
    except Exception:
        tables = []
    
    benefit_tables = []
    fee_tables = []
    
    for i, df in enumerate(tables):
        temp = df.copy()
        temp["card_name"] = card_name
        temp["cooperationcode"] = code
        temp["detail_url"] = detail_url
        temp["table_index"] = i
        
        cols_text = " ".join([str(c) for c in temp.columns])
        body_text = " ".join(temp.astype(str).fillna("").head(20).values.flatten())
        full_text = cols_text + " " + body_text
        
        if any(k in full_text for k in ["연회비", "기본연회비", "제휴연회비", "합계"]):
            fee_tables.append(temp)
        else:
            benefit_tables.append(temp)
    
    return benefit_tables, fee_tables

In [45]:
import re

def extract_benefit_text_lines_from_detail(detail_text, card_name, code, detail_url):
    lines = []
    for line in detail_text.split("\n"):
        line = re.sub(r"\s+", " ", line).strip()
        if line:
            lines.append(line)
    
    # 너무 짧은 줄 제거
    lines = [line for line in lines if len(line) >= 2]
    
    # 중복 제거
    lines = list(dict.fromkeys(lines))
    
    return pd.DataFrame({
        "card_name": [card_name] * len(lines),
        "cooperationcode": [code] * len(lines),
        "detail_url": [detail_url] * len(lines),
        "benefit_text": lines
    })

In [46]:
def parse_card_detail_html(card_name, code, session, headers):
    html, detail_url = fetch_detail_html(code, session, headers)
    
    detail_html, detail_text = extract_detail_section_before_product_desc(html)
    
    benefit_tables, fee_tables = extract_tables_from_detail_html(
        detail_html, card_name, code, detail_url
    )
    
    benefit_text_df = extract_benefit_text_lines_from_detail(
        detail_text, card_name, code, detail_url
    )
    
    summary = {
        "card_name": card_name,
        "cooperationcode": code,
        "detail_url": detail_url,
        "benefit_table_count": len(benefit_tables),
        "fee_table_count": len(fee_tables),
        "benefit_text_count": len(benefit_text_df),
    }
    summary_df = pd.DataFrame([summary])
    
    return summary_df, benefit_text_df, benefit_tables, fee_tables, detail_html

# 탭별 모두 확인 가능

In [49]:
html, detail_url = fetch_detail_html("09250", session, headers)

soup = BeautifulSoup(html, "html.parser")

for a in soup.select('div.contentArea.contentDetail a[href^="#tabCon"]'):
    print("탭명:", a.get_text(" ", strip=True), "| href:", a.get("href"))

탭명: 주요혜택 | href: #tabCon00
탭명: 상세혜택 | href: #tabCon01
탭명: 연회비 | href: #tabCon02
탭명: 확인사항 | href: #tabCon03
탭명: 적립형 선택 | href: #tabCon010
탭명: 할인형 선택 | href: #tabCon011
탭명: 7대 영역 | href: #tabCon012


In [50]:
def fetch_detail_html(code, session, headers):
    detail_url = f"https://card.kbcard.com/CRD/DVIEW/HCAMCXPRICAC0076?mainCC=a&cooperationcode={code}"
    res = session.get(detail_url, headers=headers, timeout=20)
    res.raise_for_status()
    return res.text, detail_url

In [51]:
def clean_text_lines(text):
    lines = []
    for line in text.split("\n"):
        line = re.sub(r"\s+", " ", line).strip()
        if line:
            lines.append(line)
    return lines

In [52]:
def cut_before_product_desc(target_div):
    if target_div is None:
        return None

    new_soup = BeautifulSoup("<div id='cut_area'></div>", "html.parser")
    cut_area = new_soup.select_one("#cut_area")

    for child in target_div.children:
        if isinstance(child, Tag):
            txt = child.get_text(" ", strip=True)

            if child.name == "h3" and "상품설명서" in txt:
                break

            cut_area.append(child)

    return cut_area

In [53]:
def extract_text_and_tables_from_div(target_div, card_name, code, detail_url, section_name):
    if target_div is None:
        return None, None

    cut_div = cut_before_product_desc(target_div)
    if cut_div is None:
        return None, None

    html_str = str(cut_div)
    text_str = cut_div.get_text("\n", strip=True)
    lines = clean_text_lines(text_str)

    text_df = pd.DataFrame({
        "card_name": [card_name] * len(lines),
        "cooperationcode": [code] * len(lines),
        "detail_url": [detail_url] * len(lines),
        "section_name": [section_name] * len(lines),
        "content_text": lines
    })

    table_dfs = []
    try:
        tables = pd.read_html(StringIO(html_str))
        for i, df in enumerate(tables):
            temp = df.copy()
            temp["card_name"] = card_name
            temp["cooperationcode"] = code
            temp["detail_url"] = detail_url
            temp["section_name"] = section_name
            temp["table_index"] = i
            table_dfs.append(temp)
    except Exception:
        pass

    return text_df, table_dfs

In [56]:
html, detail_url = fetch_detail_html("09250", session, headers)
soup = BeautifulSoup(html, "html.parser")

for a in soup.select('a[href^="#tabCon01"]'):
    tab_name = a.get_text(" ", strip=True)
    href = a.get("href", "").replace("#", "").strip()
    target_div = soup.find(id=href)

    print("탭명:", tab_name)
    print("href:", href)
    print("target_div 존재:", target_div is not None)
    print("-" * 50)

탭명: 상세혜택
href: tabCon01
target_div 존재: True
--------------------------------------------------
탭명: 적립형 선택
href: tabCon010
target_div 존재: True
--------------------------------------------------
탭명: 할인형 선택
href: tabCon011
target_div 존재: True
--------------------------------------------------
탭명: 7대 영역
href: tabCon012
target_div 존재: True
--------------------------------------------------


In [60]:
def parse_card_sections(card_name, code, session, headers):
    html, detail_url = fetch_detail_html(code, session, headers)
    soup = BeautifulSoup(html, "html.parser")

    all_text_dfs = []
    all_table_dfs = []

    # 1) 주요혜택
    target_div = soup.find(id="tabCon00")
    text_df, table_dfs = extract_text_and_tables_from_div(
        target_div=target_div,
        card_name=card_name,
        code=code,
        detail_url=detail_url,
        section_name="주요혜택"
    )

    if text_df is not None and not text_df.empty:
        all_text_dfs.append(text_df)

    if table_dfs:
        all_table_dfs.extend(table_dfs)

    # 2) 연회비
    target_div = soup.find(id="tabCon02")
    text_df, table_dfs = extract_text_and_tables_from_div(
        target_div=target_div,
        card_name=card_name,
        code=code,
        detail_url=detail_url,
        section_name="연회비"
    )

    if text_df is not None and not text_df.empty:
        all_text_dfs.append(text_df)

    if table_dfs:
        all_table_dfs.extend(table_dfs)

    # 3) 상세혜택 하위 탭만 수집
    subtab_links = soup.select('a[href^="#tabCon01"]')

    seen = set()
    for a in subtab_links:
        tab_name = a.get_text(" ", strip=True)
        href = a.get("href", "").replace("#", "").strip()

        # 상위 상세혜택(tabCon01) 제외
        if href == "tabCon01":
            continue

        if not tab_name or not href:
            continue

        key = (tab_name, href)
        if key in seen:
            continue
        seen.add(key)

        target_div = soup.find(id=href)
        if target_div is None:
            continue

        text_df, table_dfs = extract_text_and_tables_from_div(
            target_div=target_div,
            card_name=card_name,
            code=code,
            detail_url=detail_url,
            section_name=f"상세혜택>{tab_name}"
        )

        if text_df is not None and not text_df.empty:
            all_text_dfs.append(text_df)

        if table_dfs:
            all_table_dfs.extend(table_dfs)

    summary_df = pd.DataFrame([{
        "card_name": card_name,
        "cooperationcode": code,
        "detail_url": detail_url,
        "text_section_count": len(all_text_dfs),
        "table_section_count": len(all_table_dfs)
    }])

    return summary_df, all_text_dfs, all_table_dfs

In [61]:
summary_df, text_dfs, table_dfs = parse_card_sections(
    card_name="The Easy카드",
    code="09250",
    session=session,
    headers=headers
)

for df in text_dfs:
    print(df["section_name"].iloc[0])

주요혜택
연회비
상세혜택>적립형 선택
상세혜택>할인형 선택
상세혜택>7대 영역


In [67]:
all_summary = []
all_text = []
all_tables = []
errors = []

for idx, item in enumerate(unique_cards, start=1):
    card_name = item["card_name"]
    code = item["code"]

    try:
        summary_df, text_dfs, table_dfs = parse_card_sections(
            card_name=card_name,
            code=code,
            session=session,
            headers=headers
        )

        print(f"[{idx}/{len(unique_cards)}] 완료: {card_name} ({code})")

        all_summary.append(summary_df)

        if text_dfs:
            all_text.extend(text_dfs)

        if table_dfs:
            all_tables.extend(table_dfs)

        time.sleep(0.5)

    except Exception as e:
        print(f"에러: {card_name} ({code}) -> {e}")
        errors.append({
            "card_name": card_name,
            "cooperationcode": code,
            "error": str(e)
        })

[1/9] 완료: WE:SH All+ 카드 (09297)
[2/9] 완료: ALL 카드 (09922)
[3/9] 완료: WE:SH Daily 카드 (09570)
[4/9] 완료: 가온올림카드(실속형) (09157)
[5/9] 완료: The Easy카드 (09250)
[6/9] 완료: 마이핏카드(적립형) (09247)
[7/9] 완료: KB Pay 챌린지카드 (09113)
[8/9] 완료: KB Pay 챌린지+카드 (09114)
[9/9] 완료: toss KB국민카드 (04350)


In [68]:
summary_final = pd.concat(all_summary, ignore_index=True) if all_summary else pd.DataFrame()
text_final = pd.concat(all_text, ignore_index=True) if all_text else pd.DataFrame()
table_final = pd.concat(all_tables, ignore_index=True) if all_tables else pd.DataFrame()
error_final = pd.DataFrame(errors) if errors else pd.DataFrame()

In [69]:
summary_final.to_csv("kbcard_sections_summary.csv", index=False, encoding="utf-8-sig")
text_final.to_csv("kbcard_sections_text.csv", index=False, encoding="utf-8-sig")
table_final.to_csv("kbcard_sections_tables.csv", index=False, encoding="utf-8-sig")

if not error_final.empty:
    error_final.to_csv("kbcard_sections_errors.csv", index=False, encoding="utf-8-sig")

In [70]:
text_final["section_name"].value_counts()

section_name
연회비                    300
주요혜택                   147
상세혜택>서비스 요약            142
상세혜택>선택 할인 (택1)         78
상세혜택>7대 영역              40
상세혜택>마이핏카드(적립형)         40
상세혜택>기본/추가적립            34
상세혜택>할인형 선택             30
상세혜택>적립형 선택             30
상세혜택>자동납부 할인            21
상세혜택>분기 보너스 적립          20
상세혜택>할인                 20
상세혜택>국내 및 해외 이용 할인      19
상세혜택>토스포인트 추가적립         19
상세혜택>WE:SH 서비스          18
상세혜택>스탬프                18
상세혜택>자동이체 할인            17
상세혜택>서비스요약              13
상세혜택>출석체크               12
상세혜택>국내 및 해외 가맹점 할인     11
상세혜택>서비스 한눈에 보기         10
상세혜택>토스포인트 기본적립          7
상세혜택>리워드                 7
상세혜택>할인 캐시백              5
상세혜택>기본 할인               3
Name: count, dtype: int64

In [71]:
text_final[text_final["card_name"] == "The Easy카드"][["section_name", "content_text"]]

,section_name,content_text
608,주요혜택,주요혜택
609,주요혜택,기본 적립/할인
610,주요혜택,전 가맹점
611,주요혜택,0.7%
612,주요혜택,적립/할인
...,...,...
752,상세혜택>7대 영역,The Easy카드 추가 적립/할인 서비스는 전월 이용실적 50만원 이상 시 제공되...
753,상세혜택>7대 영역,"추가 적립/할인 대상 최상위 2개 영역 및 전표 선정 시 무이자할부 이용금액, 취소..."
754,상세혜택>7대 영역,간편결제(Pay) 서비스를 통한 이용금액은 추가 적립/할인에서 제외됩니다.
755,상세혜택>7대 영역,"7대 영역 이용 횟수 합산 값이 같을 경우 이용금액이 높은 순서대로, 영역 구분(마..."


# 컬럼명 지정하기

In [3]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup, Tag
from io import StringIO

In [4]:
headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://card.kbcard.com/"
}

session = requests.Session()

In [5]:
LIST_URL = "https://card.kbcard.com/CRD/DVIEW/HCAMCXPRICAC0047"

res = session.get(LIST_URL, headers=headers, timeout=20)
res.raise_for_status()

soup = BeautifulSoup(res.text, "html.parser")

card_items = []

for a in soup.select("a.linkDetail"):
    onclick = a.get("onclick", "")
    match = re.search(r"goDetail\('(\d+)',''\)", onclick)

    if match:
        code = match.group(1)
        title_tag = a.select_one("h3")
        card_name = title_tag.get_text(strip=True) if title_tag else None

        card_items.append({
            "card_name": card_name,
            "code": code
        })

# 중복 제거
unique_cards = []
seen = set()

for item in card_items:
    key = (item["card_name"], item["code"])
    if key not in seen:
        seen.add(key)
        unique_cards.append(item)

print("수집 대상 카드 수:", len(unique_cards))
for item in unique_cards:
    print(item)

수집 대상 카드 수: 9
{'card_name': 'WE:SH All+ 카드', 'code': '09297'}
{'card_name': 'ALL 카드', 'code': '09922'}
{'card_name': 'WE:SH Daily 카드', 'code': '09570'}
{'card_name': '가온올림카드(실속형)', 'code': '09157'}
{'card_name': 'The Easy카드', 'code': '09250'}
{'card_name': '마이핏카드(적립형)', 'code': '09247'}
{'card_name': 'KB Pay 챌린지카드', 'code': '09113'}
{'card_name': 'KB Pay 챌린지+카드', 'code': '09114'}
{'card_name': 'toss KB국민카드', 'code': '04350'}


In [6]:
def fetch_detail_html(code):
    detail_url = f"https://card.kbcard.com/CRD/DVIEW/HCAMCXPRICAC0076?mainCC=a&cooperationcode={code}"
    res = session.get(detail_url, headers=headers, timeout=20)
    res.raise_for_status()
    return res.text, detail_url

In [7]:
def cut_before_product_desc(target_div):
    if target_div is None:
        return None

    new_soup = BeautifulSoup("<div id='cut_area'></div>", "html.parser")
    cut_area = new_soup.select_one("#cut_area")

    for child in target_div.children:
        if isinstance(child, Tag):
            txt = child.get_text(" ", strip=True)

            if child.name == "h3" and "상품설명서" in txt:
                break

            cut_area.append(child)

    return cut_area

In [8]:
def extract_prev_spend(text):
    m = re.search(r"\d+\s*만원\s*이상", text)
    if m:
        return m.group().replace(" ", "")
    return None

def extract_limit(text):
    # 월 5천원 / 월 1만점 / 2만점 / 분기 1회 같은 정도만 1차로
    m = re.search(r"월\s*\d+\s*(?:천원|만원|천점|만점|원|점)", text)
    if m:
        return m.group().replace(" ", "")
    
    m = re.search(r"\d+\s*(?:천원|만원|천점|만점|원|점)", text)
    if m and ("한도" in text or "적립" in text or "할인" in text):
        return m.group().replace(" ", "")
    
    m = re.search(r"분기\s*\d+회", text)
    if m:
        return m.group().replace(" ", "")
    
    return None

In [12]:
def extract_rows_from_div(target_div, card_name, code, section_name):
    results = []

    cut_div = cut_before_product_desc(target_div)
    if cut_div is None:
        return results

    html_str = str(cut_div)

    # 1) 텍스트 먼저 항상 수집
    text = cut_div.get_text("\n", strip=True)

    for line in text.split("\n"):
        line = re.sub(r"\s+", " ", line).strip()
        if not line:
            continue

        prev_spend = extract_prev_spend(line)
        limit = extract_limit(line)

        results.append({
            "code": code,
            "card_name": card_name,
            "section_name": section_name,
            "benefit": line,
            "prev_spend": prev_spend,
            "limit": limit
        })

    # 2) 표도 있으면 추가 수집
    try:
        tables = pd.read_html(StringIO(html_str))
    except Exception:
        tables = []

    for df in tables:
        df = df.copy()

        for _, row in df.iterrows():
            values = [str(v).strip() for v in row.tolist() if pd.notna(v) and str(v).strip() != ""]
            if not values:
                continue

            benefit_text = " | ".join(values)
            prev_spend = extract_prev_spend(benefit_text)
            limit = extract_limit(benefit_text)

            results.append({
                "code": code,
                "card_name": card_name,
                "section_name": section_name,
                "benefit": benefit_text,
                "prev_spend": prev_spend,
                "limit": limit
            })

    return results

In [13]:
def parse_card_benefits(card_name, code):
    html, detail_url = fetch_detail_html(code)
    soup = BeautifulSoup(html, "html.parser")

    results = []

    # 상세혜택 하위 탭만 찾기
    subtab_links = soup.select('a[href^="#tabCon01"]')

    seen = set()

    for a in subtab_links:
        tab_name = a.get_text(" ", strip=True)
        href = a.get("href", "").replace("#", "").strip()

        # 상위 상세혜택(tabCon01)은 제외
        if href == "tabCon01":
            continue

        if not tab_name or not href:
            continue

        key = (tab_name, href)
        if key in seen:
            continue
        seen.add(key)

        target_div = soup.find(id=href)
        if target_div is None:
            continue

        section_name = f"상세혜택>{tab_name}"

        rows = extract_rows_from_div(
            target_div=target_div,
            card_name=card_name,
            code=code,
            section_name=section_name
        )

        results.extend(rows)

    return results

In [14]:
benefit_raw_df = pd.DataFrame(all_rows)
benefit_raw_df = benefit_raw_df.drop_duplicates().reset_index(drop=True)

In [15]:
test_rows = parse_card_benefits("The Easy카드", "09250")
test_df = pd.DataFrame(test_rows)

print(test_df.shape)
test_df.head(30)

(104, 6)


,code,card_name,section_name,benefit,prev_spend,limit
0,09250,The Easy카드,상세혜택>적립형 선택,상세혜택 > 적립형 선택,None,None
1,09250,The Easy카드,상세혜택>적립형 선택,적립형,None,None
2,09250,The Easy카드,상세혜택>적립형 선택,기본 적립,None,None
3,09250,The Easy카드,상세혜택>적립형 선택,국내/외 전 가맹점 0.7% 적립,None,None
4,09250,The Easy카드,상세혜택>적립형 선택,전월 실적조건 및 월 적립한도 없음,None,None
5,09250,The Easy카드,상세혜택>적립형 선택,신용카드결제일 다음 영업일 적립,None,None
6,09250,The Easy카드,상세혜택>적립형 선택,적립형,None,None
7,09250,The Easy카드,상세혜택>적립형 선택,추가 적립,None,None
8,09250,The Easy카드,상세혜택>적립형 선택,7대 영역 중,None,None
9,09250,The Easy카드,상세혜택>적립형 선택,월간 이용횟수가 가장 높은 2개 영역,None,None


In [21]:
import re
import pandas as pd

def normalize_col_name(col):
    if isinstance(col, tuple):
        col = " ".join([str(x) for x in col if str(x) != "nan"])
    col = str(col)
    col = re.sub(r"\s+", " ", col).strip()
    return col

In [22]:
def detect_column_roles(df):
    cols = [normalize_col_name(c) for c in df.columns]

    role_map = {
        "benefit": None,
        "benefit_value": None,
        "prev_spend": None,
        "limit": None
    }

    for col in cols:
        col_nospace = col.replace(" ", "")

        if role_map["benefit"] is None and any(k in col_nospace for k in ["구분", "상품서비스", "서비스", "항목", "혜택구분", "대상"]):
            role_map["benefit"] = col

        if role_map["benefit_value"] is None and any(k in col_nospace for k in ["적립률", "할인율", "혜택", "적립", "할인"]):
            role_map["benefit_value"] = col

        if role_map["prev_spend"] is None and any(k in col_nospace for k in ["전월이용실적", "이용실적", "실적조건", "전월실적", "실적"]):
            role_map["prev_spend"] = col

        if role_map["limit"] is None and any(k in col_nospace for k in ["월적립한도", "월할인한도", "적립한도", "할인한도", "한도"]):
            role_map["limit"] = col

    return role_map

In [23]:
def extract_rows_from_table_by_header(df, card_name, code, section_name):
    results = []

    df = df.copy()
    df.columns = [normalize_col_name(c) for c in df.columns]

    role_map = detect_column_roles(df)

    for _, row in df.iterrows():
        benefit = None
        benefit_value = None
        prev_spend = None
        limit = None

        if role_map["benefit"] in df.columns:
            benefit = row.get(role_map["benefit"])
        if role_map["benefit_value"] in df.columns:
            benefit_value = row.get(role_map["benefit_value"])
        if role_map["prev_spend"] in df.columns:
            prev_spend = row.get(role_map["prev_spend"])
        if role_map["limit"] in df.columns:
            limit = row.get(role_map["limit"])

        # NaN 처리
        benefit = None if pd.isna(benefit) else str(benefit).strip()
        benefit_value = None if pd.isna(benefit_value) else str(benefit_value).strip()
        prev_spend = None if pd.isna(prev_spend) else str(prev_spend).strip()
        limit = None if pd.isna(limit) else str(limit).strip()

        # 전부 비어있으면 스킵
        if not any([benefit, benefit_value, prev_spend, limit]):
            continue

        # benefit 컬럼이 없으면 행 전체를 묶어서 임시 benefit으로
        if not benefit:
            vals = [str(v).strip() for v in row.tolist() if pd.notna(v) and str(v).strip()]
            benefit = " | ".join(vals) if vals else None

        # benefit_value를 benefit에 붙여서 원문 유지
        if benefit_value and benefit_value not in benefit:
            benefit = f"{benefit} | {benefit_value}"

        results.append({
            "code": code,
            "card_name": card_name,
            "section_name": section_name,
            "benefit": benefit,
            "prev_spend": prev_spend,
            "limit": limit
        })

    return results

In [27]:
from copy import copy

def extract_rows_from_div(target_div, card_name, code, section_name):
    results = []

    cut_div = cut_before_product_desc(target_div)
    if cut_div is None:
        return results

    html_str = str(cut_div)

    # 1) 텍스트 수집용: table 제거한 뒤 텍스트만 추출
    text_soup = BeautifulSoup(html_str, "html.parser")

    for table in text_soup.select("table"):
        table.decompose()

    text_only = text_soup.get_text("\n", strip=True)

    for line in text_only.split("\n"):
        line = re.sub(r"\s+", " ", line).strip()
        if not line:
            continue

        prev_spend = extract_prev_spend(line)
        limit = extract_limit(line)

        results.append({
            "code": code,
            "card_name": card_name,
            "section_name": section_name,
            "benefit": line,
            "prev_spend": prev_spend,
            "limit": limit
        })

    # 2) 표는 표대로 따로 수집
    try:
        tables = pd.read_html(StringIO(html_str))
    except Exception:
        tables = []

    for df in tables:
        table_rows = extract_rows_from_table_by_header(
            df=df,
            card_name=card_name,
            code=code,
            section_name=section_name
        )
        results.extend(table_rows)

    return results

In [28]:
def extract_rows_from_table_by_header(df, card_name, code, section_name):
    results = []

    df = df.copy()
    df.columns = [normalize_col_name(c) for c in df.columns]

    role_map = detect_column_roles(df)

    for _, row in df.iterrows():
        benefit = None
        benefit_value = None
        prev_spend = None
        limit = None

        if role_map["benefit"] in df.columns:
            benefit = row.get(role_map["benefit"])
        if role_map["benefit_value"] in df.columns:
            benefit_value = row.get(role_map["benefit_value"])
        if role_map["prev_spend"] in df.columns:
            prev_spend = row.get(role_map["prev_spend"])
        if role_map["limit"] in df.columns:
            limit = row.get(role_map["limit"])

        benefit = None if pd.isna(benefit) else str(benefit).strip()
        benefit_value = None if pd.isna(benefit_value) else str(benefit_value).strip()
        prev_spend = None if pd.isna(prev_spend) else str(prev_spend).strip()
        limit = None if pd.isna(limit) else str(limit).strip()

        if not any([benefit, benefit_value, prev_spend, limit]):
            continue

        if not benefit:
            vals = [str(v).strip() for v in row.tolist() if pd.notna(v) and str(v).strip()]
            benefit = " | ".join(vals) if vals else None

        if benefit_value and benefit_value not in benefit:
            benefit = f"{benefit} | {benefit_value}"

        results.append({
            "code": code,
            "card_name": card_name,
            "section_name": section_name,
            "benefit": benefit,
            "prev_spend": prev_spend,
            "limit": limit
        })

    return results

In [29]:
test_rows = parse_card_benefits("The Easy카드", "09250")
test_df = pd.DataFrame(test_rows)

test_df[test_df["section_name"] == "상세혜택>적립형 선택"]

,code,card_name,section_name,benefit,prev_spend,limit
0,09250,The Easy카드,상세혜택>적립형 선택,상세혜택 > 적립형 선택,None,None
1,09250,The Easy카드,상세혜택>적립형 선택,적립형,None,None
2,09250,The Easy카드,상세혜택>적립형 선택,기본 적립,None,None
3,09250,The Easy카드,상세혜택>적립형 선택,국내/외 전 가맹점 0.7% 적립,None,None
4,09250,The Easy카드,상세혜택>적립형 선택,전월 실적조건 및 월 적립한도 없음,None,None
5,09250,The Easy카드,상세혜택>적립형 선택,신용카드결제일 다음 영업일 적립,None,None
6,09250,The Easy카드,상세혜택>적립형 선택,적립형,None,None
7,09250,The Easy카드,상세혜택>적립형 선택,추가 적립,None,None
8,09250,The Easy카드,상세혜택>적립형 선택,7대 영역 중,None,None
9,09250,The Easy카드,상세혜택>적립형 선택,월간 이용횟수가 가장 높은 2개 영역,None,None


In [30]:
df = test_df.copy()

noise_keywords = [
    "상세혜택", "적립형", "할인형",
    "구분", "적립률", "전월 이용 실적",
    "월 적립한도", "선택서비스",
    "조회 및 변경"
]

df = df[~df["benefit"].apply(lambda x: any(k in str(x) for k in noise_keywords))]

In [31]:
def is_real_benefit(text):
    text = str(text)

    # 표 형태는 무조건 살림
    if "|" in text:
        return True

    # 퍼센트/금액 포함
    if any(k in text for k in ["%", "적립", "할인", "포인트"]):
        return True

    return False

df = df[df["benefit"].apply(is_real_benefit)]

In [32]:
remove_sentences = [
    "결제일", "제외", "제공됩니다",
    "적용됩니다", "가능", "유의",
    "조회", "변경", "서비스"
]

df = df[~df["benefit"].apply(lambda x: any(k in str(x) for k in remove_sentences))]

In [33]:
df = df[df["benefit"].str.len() > 10]

In [34]:
df.head(20)

,code,card_name,section_name,benefit,prev_spend,limit
3,09250,The Easy카드,상세혜택>적립형 선택,국내/외 전 가맹점 0.7% 적립,None,None
10,09250,The Easy카드,상세혜택>적립형 선택,을 찾아 총합산 이용금액의 3% 추가 적립,None,None
11,09250,The Easy카드,상세혜택>적립형 선택,다음달 두 번째 금요일 포인트리 적립,None,None
23,09250,The Easy카드,상세혜택>할인형 선택,국내/외 전 가맹점 0.7% 할인,None,None
24,09250,The Easy카드,상세혜택>할인형 선택,전월 실적조건 및 월 할인한도 없음,None,None
30,09250,The Easy카드,상세혜택>할인형 선택,을 찾아 5% 추가 할인,None,None
